# Inference Notebook

Run inference with trained models. Loads from local checkpoint or Hugging Face Hub.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from pytorch_lightning import seed_everything

from src.data import GestureDataModule
from src.hub import get_model
from src.models import BiLSTMModule, TransformerModule

## Configuration

In [ ]:
MODEL = 'bilstm'  # 'bilstm' or 'transformer'
LOCAL_CHECKPOINT = None  # Set path or None to use get_model()

DATA_PATH = '../data/DYLEM-GRID'
TEST_SPLIT = 0.2
BATCH_SIZE = 32
SEED = 42

seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Load Model

In [ ]:
if LOCAL_CHECKPOINT:
    MODEL_CLASS = BiLSTMModule if MODEL == 'bilstm' else TransformerModule
    model = MODEL_CLASS.load_from_checkpoint(LOCAL_CHECKPOINT)
    print(f'Loaded from: {LOCAL_CHECKPOINT}')
else:
    model = get_model(MODEL)
    print(f'Loaded {MODEL.upper()} (local or Hub)')

model = model.to(device).eval()
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## Load Data

In [ ]:
dm = GestureDataModule(
    data_path=DATA_PATH,
    batch_size=BATCH_SIZE,
    test_split=TEST_SPLIT,
    seed=SEED
)
dm.setup()

print(f'Classes: {dm.class_names}')
print(f'Test samples: {len(dm.test_dataset)}')

## Run Inference

In [ ]:
all_preds, all_targets, all_probs = [], [], []

with torch.no_grad():
    for x, y in dm.test_dataloader():
        out = model(x.to(device))
        probs = torch.softmax(out, dim=1)
        preds = out.argmax(1)
        
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y.numpy())
        all_probs.extend(probs.cpu().numpy())

print(f'Inference complete: {len(all_preds)} samples')

## Results

In [ ]:
accuracy = accuracy_score(all_targets, all_preds)

print('=' * 50)
print(f'ACCURACY: {accuracy:.4f} ({accuracy*100:.2f}%)')
print('=' * 50)
print('\nClassification Report:')
print(classification_report(all_targets, all_preds, target_names=dm.class_names))

## Confusion Matrix

In [ ]:
cm = confusion_matrix(all_targets, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=dm.class_names, yticklabels=dm.class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'{MODEL.upper()} Confusion Matrix (Acc: {accuracy:.2%})')
plt.tight_layout()
plt.show()

## Per-Class Accuracy

In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1)

plt.figure(figsize=(10, 4))
bars = plt.bar(dm.class_names, per_class_acc, color='steelblue')
plt.axhline(accuracy, color='red', linestyle='--', label=f'Overall: {accuracy:.2%}')
plt.ylabel('Accuracy')
plt.title('Per-Class Accuracy')
plt.ylim(0, 1.1)
plt.legend()

for bar, acc in zip(bars, per_class_acc):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{acc:.1%}', ha='center')
plt.show()